# 📊 YOLO Visualizer & Comparison Tool

Notebook untuk visualisasi dan perbandingan hasil training YOLO.

**Fitur:**
- ✅ Load history dari `training_history.txt`
- ✅ Plot perbandingan metrik antar eksperimen
- ✅ Tabel perbandingan performa
- ✅ Analisis improvement/degradation
- ✅ Visualisasi multi-eksperimen

**Usage:**
1. Pastikan file `training_history.txt` sudah ada (hasil dari training)
2. Edit nama eksperimen di cell konfigurasi
3. Jalankan cell-by-cell untuk melihat visualisasi

## 📦 Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import ast

## 💾 History Loading Functions

In [ ]:
HISTORY_FILE = "training_history.txt"

def load_history(experiment_name, history_file=HISTORY_FILE):
    """
    Load history dari file untuk eksperimen tertentu
    
    Args:
        experiment_name: Nama eksperimen yang ingin di-load
        history_file: Path ke file history
        
    Returns:
        dict: History dictionary atau None jika tidak ditemukan
    """
    if not os.path.exists(history_file):
        print(f"❌ History file not found: {history_file}")
        return None
    
    with open(history_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Parse file untuk mencari experiment
    lines = content.split('\n')
    for line in lines:
        if line.strip().startswith(f"{experiment_name} = "):
            try:
                # Extract dictionary part
                dict_str = line.split(' = ', 1)[1]
                history = ast.literal_eval(dict_str)
                print(f"✅ Loaded experiment: {experiment_name}")
                return history
            except Exception as e:
                print(f"❌ Error parsing history: {e}")
                return None
    
    print(f"❌ Experiment '{experiment_name}' not found in history file")
    return None

def load_all_histories(history_file=HISTORY_FILE):
    """
    Load semua history dari file
    
    Returns:
        dict: Dictionary dengan key = nama eksperimen, value = history dict
    """
    if not os.path.exists(history_file):
        print(f"❌ History file not found: {history_file}")
        return {}
    
    with open(history_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    histories = {}
    lines = content.split('\n')
    
    for line in lines:
        if ' = {' in line and not line.strip().startswith('#'):
            try:
                # Extract name and dict
                parts = line.split(' = ', 1)
                if len(parts) == 2:
                    name = parts[0].strip()
                    dict_str = parts[1]
                    histories[name] = ast.literal_eval(dict_str)
            except Exception as e:
                continue
    
    print(f"✅ Loaded {len(histories)} experiments from history file")
    return histories

print("✅ History loading functions ready!")

## 📈 Visualization Functions

In [ ]:
def Plotting(history1, history2, metric='val_f1', labels=['Experiment 1', 'Experiment 2']):
    """
    Plot perbandingan metrik antara dua eksperimen
    
    Args:
        history1: Dictionary history eksperimen 1
        history2: Dictionary history eksperimen 2
        metric: Metrik yang ingin di-plot (default: 'val_f1')
        labels: Labels untuk legend
    """
    plt.figure(figsize=(12, 6))
    
    # Support berbagai format metrik
    metric_key = metric
    if metric not in history1 and metric in ['val_iou_score', 'iou', 'avg_iou']:
        # Try alternative keys
        possible_keys = ['val_mAP50_95', 'final_iou', 'val_f1']
        for key in possible_keys:
            if key in history1:
                metric_key = key
                break
    
    # Plot data
    if isinstance(history1[metric_key], list):
        epochs1 = range(1, len(history1[metric_key]) + 1)
        plt.plot(epochs1, history1[metric_key], marker='o', label=labels[0], linewidth=2)
    else:
        plt.scatter(1, history1[metric_key], s=100, label=labels[0])
    
    if isinstance(history2[metric_key], list):
        epochs2 = range(1, len(history2[metric_key]) + 1)
        plt.plot(epochs2, history2[metric_key], marker='s', label=labels[1], linewidth=2)
    else:
        plt.scatter(1, history2[metric_key], s=100, label=labels[1])
    
    plt.title(f'Model Performance Comparison - {metric}', fontsize=14, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(metric, fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def Plot_all_metrics(history, title="Training History"):
    """
    Plot semua metrik dari satu eksperimen
    
    Args:
        history: Dictionary history
        title: Judul plot
    """
    # Identify available metrics
    metrics_to_plot = []
    metric_names = []
    
    for key in ['val_f1', 'val_precision', 'val_recall', 'val_mAP50_95', 'final_iou']:
        if key in history and history[key]:
            metrics_to_plot.append(key)
            metric_names.append(key.replace('val_', '').replace('_', ' ').title())
    
    if not metrics_to_plot:
        print("❌ No metrics found to plot")
        return
    
    n_metrics = len(metrics_to_plot)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (metric, name) in enumerate(zip(metrics_to_plot[:4], metric_names[:4])):
        ax = axes[idx]
        data = history[metric]
        
        if isinstance(data, list):
            epochs = range(1, len(data) + 1)
            ax.plot(epochs, data, marker='o', linewidth=2, color=f'C{idx}')
            ax.fill_between(epochs, data, alpha=0.3, color=f'C{idx}')
        else:
            ax.bar(0, data, color=f'C{idx}', alpha=0.7)
        
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epochs', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(n_metrics, 4):
        axes[idx].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions ready!")

## 📊 Comparison Functions

In [ ]:
def Table_to_compare(value1, value2, metric_name="Metric"):
    """
    Buat tabel perbandingan antara dua nilai metrik
    
    Args:
        value1: Nilai metrik eksperimen 1
        value2: Nilai metrik eksperimen 2
        metric_name: Nama metrik
    """
    # Calculate improvement
    if isinstance(value1, list):
        value1 = value1[-1]  # Take last epoch
    if isinstance(value2, list):
        value2 = value2[-1]  # Take last epoch
    
    improvement = value2 - value1
    improvement_pct = (improvement / value1 * 100) if value1 != 0 else 0
    
    # Create comparison table
    data = {
        'Experiment': ['Baseline', 'Current', 'Improvement'],
        metric_name: [
            f"{value1:.4f}",
            f"{value2:.4f}",
            f"{improvement:+.4f} ({improvement_pct:+.2f}%)"
        ]
    }
    
    df = pd.DataFrame(data)
    
    print("\n" + "="*60)
    print(f"📊 COMPARISON TABLE - {metric_name}")
    print("="*60)
    print(df.to_string(index=False))
    print("="*60)
    
    # Analysis
    if improvement > 0:
        print(f"✅ IMPROVEMENT: Model shows {improvement_pct:.2f}% increase in {metric_name}")
    elif improvement < 0:
        print(f"⚠️ DEGRADATION: Model shows {abs(improvement_pct):.2f}% decrease in {metric_name}")
    else:
        print(f"➖ NO CHANGE: Model performance remains the same")
    print("="*60 + "\n")
    
    return df

def Compare_multiple_experiments(histories_dict, metric='final_f1'):
    """
    Bandingkan multiple eksperimen sekaligus
    
    Args:
        histories_dict: Dictionary dengan format {experiment_name: history_dict}
        metric: Metrik yang ingin dibandingkan
    """
    if not histories_dict:
        print("❌ No histories provided")
        return
    
    # Prepare data
    experiments = []
    values = []
    
    for name, history in histories_dict.items():
        experiments.append(name)
        
        # Get metric value
        if metric in history:
            val = history[metric]
            if isinstance(val, list):
                val = val[-1]
            values.append(val)
        else:
            values.append(0)
    
    # Create bar plot
    plt.figure(figsize=(12, 6))
    bars = plt.bar(experiments, values, color=plt.cm.viridis(np.linspace(0, 1, len(experiments))), alpha=0.8)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.title(f'Experiments Comparison - {metric}', fontsize=14, fontweight='bold')
    plt.xlabel('Experiments', fontsize=12)
    plt.ylabel(metric, fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    # Print table
    df = pd.DataFrame({
        'Experiment': experiments,
        metric: [f"{v:.4f}" for v in values]
    })
    print("\n" + df.to_string(index=False) + "\n")

print("✅ Comparison functions ready!")

---

## 🎯 USAGE EXAMPLES

### Example 1: Compare Two Experiments

In [ ]:
# ----- KONFIGURASI -----
# Sesuaikan nama eksperimen yang ingin dibandingkan
EXPERIMENT_1 = 'history_experiment1'  # Baseline experiment
EXPERIMENT_2 = 'history_experiment2'  # New experiment

# ----- LOAD HISTORIES -----
print("📂 Loading histories from file...\n")
history_1 = load_history(EXPERIMENT_1)
history_2 = load_history(EXPERIMENT_2)

if history_1 is None or history_2 is None:
    print("❌ One or more experiments not found!")
    print("💡 Make sure you have run training and saved the history first.")
else:
    # ----- PLOTTING COMPARISON -----
    print("\n📈 Plotting comparisons...\n")
    
    # Plot F1-Score comparison
    if 'val_f1' in history_1 and 'val_f1' in history_2:
        Plotting(history_1, history_2, metric='val_f1', 
                labels=[EXPERIMENT_1, EXPERIMENT_2])
    
    # Plot all metrics for both experiments
    if any(isinstance(v, list) for v in history_1.values()):
        Plot_all_metrics(history_1, title=f"Training History - {EXPERIMENT_1}")
        Plot_all_metrics(history_2, title=f"Training History - {EXPERIMENT_2}")
    
    # ----- TABLE COMPARISON -----
    print("\n📊 Comparing final metrics...\n")
    
    # Compare F1-Score
    if 'final_f1' in history_1 and 'final_f1' in history_2:
        Table_to_compare(history_1['final_f1'], history_2['final_f1'], 'F1-Score')
    
    # Compare Precision
    if 'final_precision' in history_1 and 'final_precision' in history_2:
        Table_to_compare(history_1['final_precision'], history_2['final_precision'], 'Precision')
    
    # Compare Recall
    if 'final_recall' in history_1 and 'final_recall' in history_2:
        Table_to_compare(history_1['final_recall'], history_2['final_recall'], 'Recall')
    
    # Compare IoU
    if 'final_iou' in history_1 and 'final_iou' in history_2:
        Table_to_compare(history_1['final_iou'], history_2['final_iou'], 'Average IoU')
    
    # ----- FINAL IMPROVEMENT SUMMARY -----
    print("\n" + "="*80)
    print("📝 FINAL IMPROVEMENT SUMMARY")
    print("="*80)
    
    metrics_to_compare = ['final_f1', 'final_precision', 'final_recall', 'final_iou']
    metric_names = ['F1-Score', 'Precision', 'Recall', 'Avg IoU']
    
    improvements = []
    for metric, name in zip(metrics_to_compare, metric_names):
        if metric in history_1 and metric in history_2:
            val1 = history_1[metric] if not isinstance(history_1[metric], list) else history_1[metric][-1]
            val2 = history_2[metric] if not isinstance(history_2[metric], list) else history_2[metric][-1]
            improvement = ((val2 - val1) / val1 * 100) if val1 != 0 else 0
            improvements.append((name, val1, val2, improvement))
            
            sign = "✅" if improvement > 0 else "⚠️" if improvement < 0 else "➖"
            print(f"{sign} {name:15s}: {val1:.4f} → {val2:.4f} ({improvement:+.2f}%)")
    
    print("="*80)
    
    # Overall assessment
    avg_improvement = np.mean([imp[3] for imp in improvements]) if improvements else 0
    if avg_improvement > 5:
        print(f"\n🎉 EXCELLENT! Overall improvement: {avg_improvement:.2f}%")
        print("The experiment shows significant performance gains across metrics.")
    elif avg_improvement > 0:
        print(f"\n👍 GOOD! Overall improvement: {avg_improvement:.2f}%")
        print("The experiment shows positive improvement.")
    elif avg_improvement > -5:
        print(f"\n⚠️ MIXED RESULTS. Overall change: {avg_improvement:.2f}%")
        print("The experiment shows minimal change. Consider adjusting hyperparameters.")
    else:
        print(f"\n❌ DEGRADATION. Overall change: {avg_improvement:.2f}%")
        print("The experiment shows performance degradation. Review your changes.")
    
    print("="*80 + "\n")

### Example 2: Compare Multiple Experiments

In [ ]:
# Load all histories from file
all_histories = load_all_histories()

if not all_histories:
    print("❌ No histories found in file!")
    print("💡 Run training first and save the history.")
else:
    print(f"✅ Found {len(all_histories)} experiments:\n")
    for name in all_histories.keys():
        print(f"   - {name}")
    print()
    
    # Compare all experiments for each metric
    metrics_to_compare = {
        'final_f1': 'F1-Score',
        'final_precision': 'Precision',
        'final_recall': 'Recall',
        'final_iou': 'Average IoU'
    }
    
    for metric_key, metric_name in metrics_to_compare.items():
        # Check if all experiments have this metric
        if all(metric_key in hist for hist in all_histories.values()):
            print(f"\n📊 Comparing: {metric_name}")
            print("-" * 60)
            Compare_multiple_experiments(all_histories, metric=metric_key)
    
    # Summary table
    print("\n" + "="*80)
    print("📋 COMPLETE EXPERIMENTS SUMMARY")
    print("="*80)
    
    summary_data = []
    for exp_name, history in all_histories.items():
        row = {'Experiment': exp_name}
        for metric_key, metric_name in metrics_to_compare.items():
            if metric_key in history:
                val = history[metric_key]
                if isinstance(val, list):
                    val = val[-1]
                row[metric_name] = f"{val:.4f}"
            else:
                row[metric_name] = "N/A"
        summary_data.append(row)
    
    df_summary = pd.DataFrame(summary_data)
    print(df_summary.to_string(index=False))
    print("="*80 + "\n")
    
    # Find best performing experiment
    for metric_key, metric_name in metrics_to_compare.items():
        if all(metric_key in hist for hist in all_histories.values()):
            best_exp = max(all_histories.items(), 
                          key=lambda x: x[1][metric_key] if not isinstance(x[1][metric_key], list) 
                                        else x[1][metric_key][-1])
            best_value = best_exp[1][metric_key]
            if isinstance(best_value, list):
                best_value = best_value[-1]
            print(f"🏆 Best {metric_name}: {best_exp[0]} ({best_value:.4f})")
    
    print()

---

## 💡 Tips

- Selalu gunakan nama eksperimen yang deskriptif (e.g., `history_baseline`, `history_with_augmentation`, dll)
- Simpan hasil training secara konsisten dengan `save_history()` function
- Gunakan notebook ini setelah training selesai untuk analisis performa
- Untuk perbandingan visual yang lebih baik, pastikan semua eksperimen menggunakan metrik yang sama